# Reto Incendios

Dos pasos separados:

| Paso | Qué hace | Red |
|---|---|---|
| **1 · Crawling** | Descarga con crawl4ai los focos de satélite, las áreas quemadas, las incidencias 112 y el riesgo de AEMET | sí |
| **2 · Datos** | Revisa cada capa en tablas, la guarda en `retos/output/incendios/<fecha>/` y comprueba que los ficheros están completos | no |

| Capa | Fuente | Clave |
|---|---|---|
| Focos de calor | NASA FIRMS (VIIRS, MODIS) · EUMETSAT LSA-SAF (SEVIRI) | SEVIRI: usuario LSA-SAF |
| Áreas quemadas | Copernicus EFFIS | no |
| Incidencias 112 | INFOCA · INFOCAM/FIDIAS · Bombers · INFORCYL | no |
| Riesgo previsto | AEMET (mapa oficial georreferenciado, 3 días) | AEMET |
| Cobertura del suelo en cada foco | ESA WorldCover (datos locales) | no (generar una vez) |

Solo produce datos, no abre ningún mapa: para verlos en el mapa interactivo está el pipeline completo (`pipeline_incendios.ipynb`).

## 0. Preparación

In [1]:
# Recarga automática: si el código de incendios/ o retos/ cambia, se usa la versión nueva sin reiniciar.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "incendios" / "__init__.py").exists()), None)
if RAIZ is None:
    raise FileNotFoundError("No encuentro la carpeta incendios/: abre el notebook desde Incendios-V0.0/ o retos/")
for ruta in (RAIZ, RAIZ / "retos"):
    if str(ruta) not in sys.path:
        sys.path.insert(0, str(ruta))

import pandas as pd

from incendios.config import fijar_claves, estado_claves
from incendios.pipeline import resumen
from reto_incendios import comprobar, crawlear, guardar, leer_json, nueva_carpeta

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("Proyecto:", RAIZ)

Proyecto: /Users/ivanduartepacheco/Downloads/trabajo/proyecto/Incendios-V0.0


## 1. Claves

- **AEMET:** necesaria para el riesgo previsto.
- **LSA-SAF:** necesaria para los focos de Meteosat SEVIRI.
- **Sin ellas:** FIRMS, EFFIS y las incidencias 112 funcionan igual.

Si ya están en `.env` (por ejemplo, porque las guardaste desde el pipeline), deja las comillas vacías. ⚠️ Si las pegas aquí, no subas este notebook a git.

In [2]:
AEMET_API_KEY   = ""
LSASAF_USUARIO  = ""   # opcional (SEVIRI)
LSASAF_PASSWORD = ""   # opcional (SEVIRI)

fijar_claves(aemet=AEMET_API_KEY, lsasaf_usuario=LSASAF_USUARIO, lsasaf_password=LSASAF_PASSWORD)
for variable, valor in estado_claves().items():
    print(f"{variable:<16} {valor}")

AEMET_API_KEY    eyJh…EFOA (284 caracteres)
FIRMS_MAP_KEY    — (vacía)
LSASAF_USER      ivan…te19 (12 caracteres)
LSASAF_PASSWORD  ****


## 2. Parámetros

In [3]:
HORAS_FOCOS      = 24    # ventana de focos FIRMS (24, 48 o hasta 168 h)
HORAS_SEVIRI     = 3     # ventana de focos SEVIRI (4 ficheros por hora)
DIAS_EFFIS       = 7     # áreas quemadas publicadas o revisadas
DIAS_INCIDENCIAS = 3     # incidencias 112 ya extinguidas que se incluyen

# PASO 1 · Crawling

Las cinco fuentes se lanzan en paralelo con crawl4ai. Cada una va aislada: si falla o le falta la clave, queda anotada como `error` u `omitida` y las demás siguen.

### 1.1 Ejecutar el crawling

In [4]:
parametros = {"horas_focos": HORAS_FOCOS, "horas_seviri": HORAS_SEVIRI,
              "dias_effis": DIAS_EFFIS, "dias_incidencias": DIAS_INCIDENCIAS}
resultado = await crawlear(**parametros)
print(resumen(resultado["estado"]))

[INIT].... → Crawl4AI 0.8.9 

[INIT].... → Crawl4AI 0.8.9 

[INIT].... → Crawl4AI 0.8.9 

[INIT].... → Crawl4AI 0.8.9 

[INIT].... → Crawl4AI 0.8.9 

[INIT].... → Crawl4AI 0.8.9 

  ok       Incidencias 112 (oficiales)          14 registros   10.1s
  ok       EFFIS (áreas quemadas)               26 registros   10.6s
  ok       Riesgo de incendio (AEMET)            3 registros   11.6s
  ok       NASA FIRMS                           24 registros   11.8s
  ok       EUMETSAT LSA-SAF (SEVIRI)             0 registros   16.9s
                                            vigiló el 3.1% de España (76.8% bajo nubes, 13:30 UTC)


# PASO 2 · Datos

Revisión de cada capa, guardado en ficheros (GeoJSON, imágenes de riesgo, `estado.json` y `manifiesto.json`) y comprobación de que se han escrito completos. No hace peticiones de red: trabaja con lo que ha descargado el paso 1.

### 2.1 Focos de calor

Color por confianza (alta/media/baja) y `frp_mw` = potencia radiativa del fuego. SEVIRI no ve a través de las nubes: el resumen de arriba indica qué porcentaje de España pudo vigilar.

In [5]:
df_focos = pd.DataFrame(resultado["focos"])
if df_focos.empty:
    print("Sin focos en la ventana.")
else:
    display(df_focos.groupby(["sensor", "satelite", "confianza"]).size().rename("focos").to_frame())
    display(df_focos.sort_values("frp_mw", ascending=False)[["fecha_utc", "satelite", "lat", "lon", "confianza", "frp_mw"]].head(10))

focos
sensor satelite  confianza       
MODIS  Aqua      baja           1
       Terra     baja           3
                 media          2
VIIRS  NOAA-20   media          7
       NOAA-21   media          5
       Suomi NPP media          6

,fecha_utc,satelite,lat,lon,confianza,frp_mw
22,2026-10-06T15:52:00+00:00,Aqua,38.58708,-5.22434,baja,14.33
4,2026-10-07T09:49:00+00:00,Terra,37.37804,-5.05025,media,14.07
3,2026-10-07T09:49:00+00:00,Terra,38.25338,-1.61175,baja,10.10
2,2026-10-07T09:49:00+00:00,Terra,39.62090,-7.45476,baja,9.43
0,2026-10-07T09:49:00+00:00,Terra,41.42020,-1.37181,baja,8.69
1,2026-10-07T09:49:00+00:00,Terra,39.64728,-6.96796,media,6.68
5,2026-10-07T03:10:00+00:00,NOAA-21,40.24611,-3.47442,media,4.10
14,2026-10-07T02:06:00+00:00,Suomi NPP,40.24523,-3.47569,media,3.63
23,2026-10-06T14:08:00+00:00,NOAA-20,40.24595,-3.47153,media,2.39
20,2026-10-07T01:30:00+00:00,NOAA-21,36.96750,-1.90390,media,1.73


### 2.2 Cobertura del suelo en cada foco

El paso 1 anota en cada foco y en cada incidencia qué hay en el suelo a su alrededor (ESA WorldCover, datos locales). Así se separa la vegetación forestal de los cultivos, las zonas urbanas o el agua. Si la fuente aparece `omitida` en el resumen, hay que generar los datos una vez con `python herramientas/generar_cobertura.py`.

In [ ]:
if "cobertura" in df_focos:
    con = df_focos[df_focos["cobertura"].notna()]
    tabla_cob = pd.DataFrame({"fecha_utc": con["fecha_utc"], "satelite": con["satelite"], "frp_mw": con["frp_mw"],
                              "suelo dominante": con["cobertura"].map(lambda c: c["dominante"]),
                              "vegetación natural %": con["cobertura"].map(lambda c: c["combustible_pct"])})
    display(tabla_cob.sort_values("frp_mw", ascending=False))
    display(tabla_cob["suelo dominante"].value_counts().rename("focos").to_frame())
else:
    print("Sin cobertura del suelo en esta ejecución (ver el resumen del paso 1).")

### 2.3 Áreas quemadas (EFFIS)

Cuenta lo que EFFIS ha **publicado o revisado** en la ventana. La columna `novedad` distingue:
- **`nueva`:** el incendio cae dentro de la ventana.
- **`actualizada`:** incendio anterior cuyo perímetro EFFIS ha revisado.

In [6]:
df_areas = pd.DataFrame(resultado["areas"])
if df_areas.empty:
    print("EFFIS no ha publicado áreas en la ventana.")
else:
    print(df_areas["novedad"].value_counts().to_dict(), "·", df_areas["area_ha"].sum(), "ha")
    display(df_areas[["fecha_incendio", "novedad", "provincia", "municipio", "area_ha", "pct_natura2000"]]
            .sort_values("area_ha", ascending=False).head(10))

{'actualizada': 26} · 196.0 ha


,fecha_incendio,novedad,provincia,municipio,area_ha,pct_natura2000
22,2026-09-28T01:34:00+02:00,actualizada,Toledo,Lominchar,33.0,0.0
2,2026-09-21T09:13:00+02:00,actualizada,Zamora,Muelas del Pan,26.0,0.0
20,2026-09-25T12:14:00+02:00,actualizada,Segovia,Ventosilla y Tejadilla,21.0,0.0
24,2026-09-26T13:36:00+02:00,actualizada,Asturias,Lena,18.0,100.0
21,2026-09-28T01:54:00+02:00,actualizada,Toledo,Chozas de Canales,14.0,0.0
1,2026-09-28T19:57:00+02:00,actualizada,Cantabria,Campoo de Enmedio,10.0,0.0
23,2026-09-25T01:54:00+02:00,actualizada,Ourense,Manzaneda,9.0,100.0
8,2026-09-24T20:36:00+02:00,actualizada,Cantabria,"Tojos, Los",9.0,100.0
9,2026-09-24T20:36:00+02:00,actualizada,Cantabria,"Tojos, Los",9.0,100.0
25,2026-09-26T01:34:00+02:00,actualizada,Asturias,Lena,7.0,100.0


### 2.4 Incidencias 112

Las publican los servicios de emergencia autonómicos. Un "aviso sin fase" significa que el servicio ha abierto el aviso pero no lo ha clasificado; no confirma un incendio activo.

In [7]:
print(resultado["estado"]["Incidencias 112 (oficiales)"].get("detalle_fuentes"))
df_inc = pd.DataFrame(resultado["incidencias"])
if not df_inc.empty:
    display(df_inc[["comunidad", "municipio", "provincia", "estado", "inicio_utc", "superficie_ha", "fuente"]])

{'Andalucía (INFOCA)': 10, 'Castilla-La Mancha (FIDIAS)': 0, 'Cataluña (Bombers)': 0, 'Castilla y León (INFORCYL)': 4}


,comunidad,municipio,provincia,estado,inicio_utc,superficie_ha,fuente
0,Castilla y León,ALBERCA (LA),SALAMANCA,NaN,2026-10-07T13:26+00:00,None,INFORCYL
1,Castilla y León,MARAZOLEJA,SEGOVIA,NaN,2026-10-07T13:19+00:00,None,INFORCYL
2,Castilla y León,MUÑOGRANDE,AVILA,NaN,2026-10-07T12:22+00:00,None,INFORCYL
3,Castilla y León,CANDELEDA,AVILA,NaN,2026-10-07T09:40+00:00,None,INFORCYL
4,Andalucía,Casares,MÁLAGA,extinguido,2026-10-06T19:40+00:00,None,INFOCA
5,Andalucía,Paterna del Campo,HUELVA,extinguido,2026-10-06T17:05+00:00,None,INFOCA
6,Andalucía,Calañas,HUELVA,extinguido,2026-10-06T13:40+00:00,None,INFOCA
7,Andalucía,El Almendro,HUELVA,extinguido,2026-10-06T13:00+00:00,None,INFOCA
8,Andalucía,Alhaurín el Grande,MÁLAGA,extinguido,2026-10-05T20:44+00:00,None,INFOCA
9,Andalucía,Guillena,SEVILLA,extinguido,2026-10-05T05:54+00:00,None,INFOCA


### 2.5 Riesgo previsto (AEMET)

In [8]:
for r in resultado["riesgo"]:
    print(f"{r['fecha']}: nivel máximo {r['nivel_max']} · reparto (%) {r['reparto_pct']}")
if not resultado["riesgo"]:
    print("Sin riesgo: falta la clave de AEMET o la fuente falló (ver resumen).")

2026-10-08: nivel máximo Alto · reparto (%) {'Muy bajo': 85.2, 'Bajo': 14.1, 'Moderado': 0.6, 'Alto': 0.0, 'Muy alto': 0.0, 'Extremo': 0.0}
2026-10-09: nivel máximo Alto · reparto (%) {'Muy bajo': 75.1, 'Bajo': 23.8, 'Moderado': 1.1, 'Alto': 0.0, 'Muy alto': 0.0, 'Extremo': 0.0}
2026-10-10: nivel máximo Moderado · reparto (%) {'Muy bajo': 69.7, 'Bajo': 30.2, 'Moderado': 0.1, 'Alto': 0.0, 'Muy alto': 0.0, 'Extremo': 0.0}


### 2.6 Guardar los ficheros

Escribe en una carpeta nueva los GeoJSON, las imágenes de riesgo, `estado.json` y `manifiesto.json`. El manifiesto indica qué fichero es cada capa y cuántos registros tiene, para que cualquier otra aplicación pueda leerlos.

In [9]:
carpeta = nueva_carpeta("incendios")
guardar(resultado, carpeta, parametros)
print("Guardado en", carpeta)
manifiesto = leer_json(carpeta / "manifiesto.json")
display(pd.DataFrame(manifiesto["capas"]).T[["fichero", "tipo", "registros"]])

Guardado en /Users/ivanduartepacheco/Downloads/trabajo/proyecto/Incendios-V0.0/retos/output/incendios/20261007T135618Z


,fichero,tipo,registros
focos,focos.geojson,geojson,24
areas_quemadas,areas_quemadas.geojson,geojson,26
incidencias_112,incidencias_112.geojson,geojson,14
riesgo,riesgo/riesgo.json,imagen,3


### 2.7 Comprobar los ficheros guardados

Se relee lo que se acaba de escribir y se compara con el manifiesto y con lo que hay en memoria: cada fichero tiene que existir, poder leerse y tener todos sus registros.
- **Todo ✓:** los datos están completos.
- **Algún ✗:** vuelve a ejecutar la sección anterior.

In [10]:
tabla = pd.DataFrame(comprobar(carpeta))
tabla["ok"] = tabla["ok"].map({True: "✓", False: "✗"})
display(tabla)
print("Datos completos ✓" if (tabla["ok"] == "✓").all() else "Hay ficheros incompletos ✗: vuelve a guardar")
print()
print("Carpeta de datos:", carpeta)
for f in sorted(carpeta.rglob("*")):
    if f.is_file():
        print(f"   {f.relative_to(carpeta)}  ({f.stat().st_size // 1024} KB)")

,capa,fichero,manifiesto,en el fichero,ok
0,focos,focos.geojson,24,24,✓
1,areas_quemadas,areas_quemadas.geojson,26,26,✓
2,incidencias_112,incidencias_112.geojson,14,14,✓
3,riesgo,riesgo/riesgo.json,3,3,✓


Datos completos ✓

Carpeta de datos: /Users/ivanduartepacheco/Downloads/trabajo/proyecto/Incendios-V0.0/retos/output/incendios/20261007T135618Z
   areas_quemadas.geojson  (132 KB)
   estado.json  (0 KB)
   focos.geojson  (7 KB)
   incidencias_112.geojson  (5 KB)
   manifiesto.json  (1 KB)
   riesgo/niveles_D1.png  (15 KB)
   riesgo/niveles_D2.png  (19 KB)
   riesgo/niveles_D3.png  (22 KB)
   riesgo/riesgo.json  (2 KB)
   riesgo/riesgo_D1.png  (25 KB)
   riesgo/riesgo_D2.png  (31 KB)
   riesgo/riesgo_D3.png  (34 KB)


### Mapa interactivo

Genera `mapa_incendios.html` en la carpeta de datos y lo abre en el navegador. Lee solo los ficheros guardados: no hace peticiones.

In [ ]:
from reto_incendios import mapa
ruta = mapa(carpeta, abrir=True)
print("Mapa:", ruta)